# Ejercicio 5: El Peor Escenario (Stress-Test)

Comparación de **Greedy**, **A*** y búsqueda con **pesos** en escenarios adversos.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import heapq
import time
from pathlib import Path
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    ip.run_line_magic('matplotlib', 'inline')
plt.ioff()

: 

## 1. Funciones auxiliares

In [ ]:
def obtener_vecinos(nodo, mapa):
    filas, columnas = mapa.shape
    r, c = nodo
    candidatos = [
        (r, c + 1),  # derecha
        (r + 1, c),  # abajo
        (r - 1, c),  # arriba
        (r, c - 1),  # izquierda
    ]
    vecinos = []
    for nr, nc in candidatos:
        if 0 <= nr < filas and 0 <= nc < columnas and mapa[nr, nc] == 0:
            vecinos.append((nr, nc))
    return vecinos

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def reconstruir_camino(padres, fin):
    if fin not in padres:
        return []
    camino = []
    nodo = fin
    while nodo is not None:
        camino.append(nodo)
        nodo = padres.get(nodo)
    camino.reverse()
    return camino

def ejecutar_astar(mapa, inicio, fin):
    inicio_t = time.perf_counter()
    frontera = []
    contador = 0
    heapq.heappush(frontera, (manhattan(inicio, fin), contador, inicio))

    costo = {inicio: 0}
    padres = {inicio: None}
    explorados = set()
    orden = []
    pasos = 0

    while frontera:
        pasos += 1
        _, _, actual = heapq.heappop(frontera)
        if actual in explorados:
            continue

        explorados.add(actual)
        orden.append(actual)

        if actual == fin:
            break

        for vecino in obtener_vecinos(actual, mapa):
            nuevo_g = costo[actual] + 1
            if vecino not in costo or nuevo_g < costo[vecino]:
                costo[vecino] = nuevo_g
                prioridad = nuevo_g + manhattan(vecino, fin)
                contador += 1
                heapq.heappush(frontera, (prioridad, contador, vecino))
                padres[vecino] = actual

    fin_t = time.perf_counter()
    camino = reconstruir_camino(padres, fin)

    return {
        "nombre": "A*",
        "explorados": explorados,
        "orden": orden,
        "camino": camino,
        "num_explorados": len(explorados),
        "pasos": pasos,
        "tiempo": fin_t - inicio_t,
        "distancia": len(camino) - 1 if camino else None
    }

def ejecutar_greedy(mapa, inicio, fin):
    inicio_t = time.perf_counter()
    frontera = []
    contador = 0
    heapq.heappush(frontera, (manhattan(inicio, fin), contador, inicio))

    padres = {inicio: None}
    explorados = set()
    orden = []
    pasos = 0

    while frontera:
        pasos += 1
        _, _, actual = heapq.heappop(frontera)
        if actual in explorados:
            continue

        explorados.add(actual)
        orden.append(actual)

        if actual == fin:
            break

        for vecino in obtener_vecinos(actual, mapa):
            if vecino in explorados:
                continue
            if vecino not in padres:
                padres[vecino] = actual
            contador += 1
            heapq.heappush(frontera, (manhattan(vecino, fin), contador, vecino))

    fin_t = time.perf_counter()
    camino = reconstruir_camino(padres, fin)

    return {
        "nombre": "Greedy",
        "explorados": explorados,
        "orden": orden,
        "camino": camino,
        "num_explorados": len(explorados),
        "pasos": pasos,
        "tiempo": fin_t - inicio_t,
        "distancia": len(camino) - 1 if camino else None
    }

def dibujar_mapa(ax, mapa, resultado, inicio, fin, titulo, color_base):
    filas, columnas = mapa.shape
    imagen = np.ones((filas, columnas, 3))
    imagen[:, :] = [0.97, 0.97, 0.98]

    for r in range(filas):
        for c in range(columnas):
            if mapa[r, c] == 1:
                imagen[r, c] = [0.22, 0.22, 0.27]

    total = len(resultado["orden"])
    for i, (r, c) in enumerate(resultado["orden"]):
        t = i / max(total - 1, 1)
        rr = max(color_base[0] - 0.30 * t, 0.1)
        gg = max(color_base[1] - 0.30 * t, 0.1)
        bb = max(color_base[2] - 0.15 * t, 0.2)
        imagen[r, c] = [rr, gg, bb]

    for r, c in resultado["camino"]:
        imagen[r, c] = [1.0, 0.56, 0.10]

    imagen[inicio[0], inicio[1]] = [0.18, 0.80, 0.44]
    imagen[fin[0], fin[1]] = [0.92, 0.20, 0.25]

    ax.imshow(imagen, interpolation="nearest")
    ax.set_xticks(np.arange(-0.5, columnas, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, filas, 1), minor=True)
    ax.grid(which="minor", color="#cfcfcf", linewidth=0.3)
    ax.tick_params(which="minor", size=0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titulo, fontsize=11, fontweight="bold", color="#2c3e50")

print("Funciones listas.")

## 2. Escenario trampa (Greedy vs A*)

In [ ]:
def crear_laberinto_falso():
    filas, columnas = 30, 30
    mapa = np.ones((filas, columnas), dtype=int)

    inicio = (15, 2)
    fin = (2, 20)

    # Camino largo (engañoso) que va primero hacia la derecha,
    # parece acercarse al objetivo pero obliga a un gran rodeo.
    mapa[15, 2:28] = 0
    mapa[2:16, 27] = 0
    mapa[2, 20:28] = 0

    # Camino corto (óptimo) que al inicio no parece tan atractivo
    # para Greedy porque no avanza directamente a la meta.
    mapa[2:16, 2] = 0
    mapa[2, 2:21] = 0

    mapa[inicio] = 0
    mapa[fin] = 0
    return mapa, inicio, fin

mapa_trampa, inicio_t, fin_t = crear_laberinto_falso()
res_greedy = ejecutar_greedy(mapa_trampa, inicio_t, fin_t)
res_astar = ejecutar_astar(mapa_trampa, inicio_t, fin_t)

fig, ejes = plt.subplots(1, 2, figsize=(15, 7))
fig.patch.set_facecolor("#fafafa")
dibujar_mapa(
    ejes[0], mapa_trampa, res_greedy, inicio_t, fin_t,
    f"Greedy — explorados: {res_greedy['num_explorados']} | distancia: {res_greedy['distancia']}",
    [0.95, 0.67, 0.72]
)
dibujar_mapa(
    ejes[1], mapa_trampa, res_astar, inicio_t, fin_t,
    f"A* — explorados: {res_astar['num_explorados']} | distancia: {res_astar['distancia']}",
    [0.45, 0.75, 0.95]
)

leyendas = [
    mpatches.Patch(color=[0.18, 0.80, 0.44], label="Inicio"),
    mpatches.Patch(color=[0.92, 0.20, 0.25], label="Fin"),
    mpatches.Patch(color=[1.0, 0.56, 0.10], label="Camino"),
    mpatches.Patch(color=[0.22, 0.22, 0.27], label="Muro"),
]
fig.legend(handles=leyendas, loc="lower center", ncol=4, fontsize=10, frameon=True)
plt.suptitle("Escenario trampa: callejón engañoso cerca del objetivo", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
Path("img").mkdir(exist_ok=True)
plt.savefig("img/ej5_trampa_compare.png", dpi=150, bbox_inches="tight", facecolor="#fafafa")
plt.show()

print("Imagen 1 guardada: img/ej5_trampa_compare.png")
print(f"Greedy: distancia={res_greedy['distancia']} | explorados={res_greedy['num_explorados']}")
print(f"A*:     distancia={res_astar['distancia']} | explorados={res_astar['num_explorados']}")

In [ ]:
metricas = ["Distancia", "Nodos explorados", "Pasos", "Tiempo (ms)"]
val_g = [
    res_greedy["distancia"],
    res_greedy["num_explorados"],
    res_greedy["pasos"],
    res_greedy["tiempo"] * 1000
]
val_a = [
    res_astar["distancia"],
    res_astar["num_explorados"],
    res_astar["pasos"],
    res_astar["tiempo"] * 1000
]

x = np.arange(len(metricas))
ancho = 0.34

fig, ax = plt.subplots(figsize=(10.5, 5.2))
fig.patch.set_facecolor("#fafafa")
ax.set_facecolor("#fafafa")

b1 = ax.bar(x - ancho / 2, val_g, ancho, label="Greedy", color="#e74c3c", edgecolor="#c0392b")
b2 = ax.bar(x + ancho / 2, val_a, ancho, label="A*", color="#3498db", edgecolor="#2980b9")

for barras, color in [(b1, "#c0392b"), (b2, "#2980b9")]:
    for barra in barras:
        alto = barra.get_height()
        ax.annotate(f"{alto:.1f}" if alto < 10 else f"{int(round(alto))}",
                    (barra.get_x() + barra.get_width() / 2, alto),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=10, color=color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(metricas, fontsize=10)
ax.set_title("Comparación cuantitativa: Greedy vs A*", fontsize=13, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig("img/ej5_trampa_barras.png", dpi=150, bbox_inches="tight", facecolor="#fafafa")
plt.show()

print("Imagen 2 guardada: img/ej5_trampa_barras.png")

## 3. Escenario con pesos (río/pantano) — Dijkstra vs A*

In [ ]:
def crear_mapa_pesos(filas=30, columnas=30):
    pesos = np.ones((filas, columnas), dtype=float)

    # Pantano vertical de alto costo
    pesos[:, 12:18] = 8.0

    # Un puente/corredor barato para rodear
    pesos[24, 12:18] = 1.0

    inicio = (5, 3)
    fin = (5, 26)
    return pesos, inicio, fin

def obtener_vecinos_pesos(nodo, pesos):
    filas, columnas = pesos.shape
    r, c = nodo
    vecinos = []
    for nr, nc in [(r+1,c), (r-1,c), (r,c+1), (r,c-1)]:
        if 0 <= nr < filas and 0 <= nc < columnas:
            vecinos.append((nr, nc))
    return vecinos

def ejecutar_dijkstra_pesos(pesos, inicio, fin):
    inicio_t = time.perf_counter()
    frontera = []
    contador = 0
    heapq.heappush(frontera, (0.0, contador, inicio))

    costo = {inicio: 0.0}
    padres = {inicio: None}
    explorados = set()
    orden = []

    while frontera:
        _, _, actual = heapq.heappop(frontera)
        if actual in explorados:
            continue
        explorados.add(actual)
        orden.append(actual)

        if actual == fin:
            break

        for vecino in obtener_vecinos_pesos(actual, pesos):
            nuevo_g = costo[actual] + pesos[vecino]
            if vecino not in costo or nuevo_g < costo[vecino]:
                costo[vecino] = nuevo_g
                contador += 1
                heapq.heappush(frontera, (nuevo_g, contador, vecino))
                padres[vecino] = actual

    fin_t = time.perf_counter()
    camino = reconstruir_camino(padres, fin)
    costo_camino = costo.get(fin, None)
    return {
        "nombre": "Dijkstra",
        "explorados": explorados,
        "orden": orden,
        "camino": camino,
        "num_explorados": len(explorados),
        "costo": costo_camino,
        "distancia": len(camino) - 1 if camino else None,
        "tiempo": fin_t - inicio_t
    }

def ejecutar_astar_pesos(pesos, inicio, fin):
    inicio_t = time.perf_counter()
    frontera = []
    contador = 0

    min_peso = float(np.min(pesos))
    h0 = manhattan(inicio, fin) * min_peso
    heapq.heappush(frontera, (h0, contador, inicio))

    costo = {inicio: 0.0}
    padres = {inicio: None}
    explorados = set()
    orden = []

    while frontera:
        _, _, actual = heapq.heappop(frontera)
        if actual in explorados:
            continue
        explorados.add(actual)
        orden.append(actual)

        if actual == fin:
            break

        for vecino in obtener_vecinos_pesos(actual, pesos):
            nuevo_g = costo[actual] + pesos[vecino]
            if vecino not in costo or nuevo_g < costo[vecino]:
                costo[vecino] = nuevo_g
                heur = manhattan(vecino, fin) * min_peso
                prioridad = nuevo_g + heur
                contador += 1
                heapq.heappush(frontera, (prioridad, contador, vecino))
                padres[vecino] = actual

    fin_t = time.perf_counter()
    camino = reconstruir_camino(padres, fin)
    costo_camino = costo.get(fin, None)
    return {
        "nombre": "A*",
        "explorados": explorados,
        "orden": orden,
        "camino": camino,
        "num_explorados": len(explorados),
        "costo": costo_camino,
        "distancia": len(camino) - 1 if camino else None,
        "tiempo": fin_t - inicio_t
    }

def dibujar_pesos(ax, pesos, resultado, inicio, fin, titulo):
    filas, columnas = pesos.shape
    imagen = np.zeros((filas, columnas, 3))

    # Suelo normal
    imagen[:, :] = [0.93, 0.95, 0.97]

    # Pantano
    mask_pantano = pesos >= 8
    imagen[mask_pantano] = [0.42, 0.30, 0.18]

    # Puente
    for c in range(columnas):
        if pesos[24, c] == 1.0 and 12 <= c <= 17:
            imagen[24, c] = [0.86, 0.78, 0.62]

    total = len(resultado["orden"])
    for i, (r, c) in enumerate(resultado["orden"]):
        t = i / max(total - 1, 1)
        imagen[r, c] = [0.60 - 0.30*t, 0.78 - 0.30*t, 0.95 - 0.20*t]

    for r, c in resultado["camino"]:
        imagen[r, c] = [1.0, 0.55, 0.10]

    imagen[inicio] = [0.18, 0.80, 0.44]
    imagen[fin] = [0.92, 0.20, 0.25]

    ax.imshow(imagen, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titulo, fontsize=11, fontweight="bold", color="#2c3e50")

pesos, inicio_p, fin_p = crear_mapa_pesos()
res_dij = ejecutar_dijkstra_pesos(pesos, inicio_p, fin_p)
res_astar_p = ejecutar_astar_pesos(pesos, inicio_p, fin_p)

fig, ejes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor("#fafafa")
dibujar_pesos(
    ejes[0], pesos, res_dij, inicio_p, fin_p,
    f"Dijkstra — costo={res_dij['costo']:.1f} | dist={res_dij['distancia']} | nodos={res_dij['num_explorados']}"
)
dibujar_pesos(
    ejes[1], pesos, res_astar_p, inicio_p, fin_p,
    f"A* — costo={res_astar_p['costo']:.1f} | dist={res_astar_p['distancia']} | nodos={res_astar_p['num_explorados']}"
)

leyendas = [
    mpatches.Patch(color=[0.42, 0.30, 0.18], label="Pantano (costo 8)"),
    mpatches.Patch(color=[0.86, 0.78, 0.62], label="Puente limpio (costo 1)"),
    mpatches.Patch(color=[1.0, 0.55, 0.10], label="Camino final"),
]
fig.legend(handles=leyendas, loc="lower center", ncol=3, fontsize=10, frameon=True)
plt.suptitle("Escenario con pesos: línea recta cruza pantano", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig("img/ej5_pantano_compare.png", dpi=150, bbox_inches="tight", facecolor="#fafafa")
plt.show()

pantano_mask = pesos >= 8
explorados_pantano_dij = sum(1 for n in res_dij['explorados'] if pantano_mask[n])
explorados_pantano_ast = sum(1 for n in res_astar_p['explorados'] if pantano_mask[n])

print("Imagen 3 guardada: img/ej5_pantano_compare.png")
print(f"Dijkstra: costo={res_dij['costo']:.1f}, distancia={res_dij['distancia']}, explorados={res_dij['num_explorados']}")
print(f"A*:      costo={res_astar_p['costo']:.1f}, distancia={res_astar_p['distancia']}, explorados={res_astar_p['num_explorados']}")
print(f"Nodos del pantano explorados -> Dijkstra: {explorados_pantano_dij}, A*: {explorados_pantano_ast}")

## 4. Preguntas de discusión (resueltas)

### 1) ¿En qué casos específicos Greedy termina siendo "más rápido" pero con una solución desastrosa en distancia?
Greedy puede ser más rápido cuando el objetivo está en línea recta y la heurística lo empuja con fuerza a esa zona, pero existe un bloqueo tardío (callejón sin salida o muro largo al final). En ese caso encuentra *rápido* una ruta, pero la ruta puede ser mucho más larga porque no optimiza el costo acumulado.

### 2) ¿Existe algún escenario donde A* visite más nodos que BFS?
Sí, puede pasar si la heurística es muy débil (casi constante o muy poco informativa). En ese límite A* se comporta parecido a UCS y puede expandir una cantidad de nodos comparable o incluso mayor que BFS dependiendo de desempates y estructura del mapa.

### 3) ¿A* intentó forzar la entrada al pantano antes de rendirse?
En este experimento se observa que A* sí evalúa parte de la zona de alto costo cercana a la línea recta, pero al combinar `g(n)+h(n)` termina favoreciendo el rodeo por el puente limpio, reduciendo exploración inútil respecto a una expansión más uniforme.

### 4) ¿Cómo afecta el peso de las celdas a la función f(n)=g(n)+h(n)?
El peso impacta directamente `g(n)`: entrar en celdas caras eleva mucho el costo acumulado. Por eso, aunque `h(n)` siga apuntando a la meta, un `g(n)` grande puede volver menos atractiva la línea recta y desplazar la búsqueda hacia rutas más largas en pasos pero más baratas en costo total.

### 5) Si aumentamos el peso del pantano al infinito, ¿en qué algoritmo se convierte Dijkstra?
Dijkstra no cambia de algoritmo: sigue siendo Dijkstra/UCS. Lo que cambia es el grafo efectivo: las celdas con costo infinito se vuelven intransitables (equivalentes a obstáculos), y el algoritmo solo considera rutas por celdas con costo finito.

In [ ]:
print("Resumen numérico final:")
print(f"Greedy -> distancia={res_greedy['distancia']}, explorados={res_greedy['num_explorados']}, tiempo_ms={res_greedy['tiempo']*1000:.3f}")
print(f"A* (trampa) -> distancia={res_astar['distancia']}, explorados={res_astar['num_explorados']}, tiempo_ms={res_astar['tiempo']*1000:.3f}")
print(f"Dijkstra (pesos) -> costo={res_dij['costo']:.1f}, distancia={res_dij['distancia']}, explorados={res_dij['num_explorados']}")
print(f"A* (pesos) -> costo={res_astar_p['costo']:.1f}, distancia={res_astar_p['distancia']}, explorados={res_astar_p['num_explorados']}")